In [5]:
"""
TOP1 高斯过程模型 (Gp_GP_Matern_0.5_K6_Comb17) 完整学习曲线分析程序 v4
======================================================================
v4 在 v3 基础上新增五大类补充分析（共新增 8 张图，Excel 新增 10 个工作表）：

  A. 方差与稳定性分析
     图4: Box-whisker + Violin 双联图（RMSE/MAE/R² 方差诊断）
     图5: 变异系数 CV(%) 曲线（Train vs Val，稳定性演变）

  B. 预测质量深度分析
     图6: 残差分布演变图（KDE + 偏度/峰度标注，4个代表性训练集大小）
     图7: GP 不确定性校准曲线（Calibration: Expected coverage vs Observed coverage）
     图8: PI 实际覆盖率 + PICP-MPIW 联合图

  C. 收敛分析
     图9:  幂律拟合曲线（RMSE = a·nᵇ，含收敛速率指数与预测）
            + 边际收益递减图（ΔRMSE/ΔN）
     图10: Area Between Curves（泛化代价积分） + Bootstrap CI 置信带

  D. GP 专属诊断
     图11: 核超参数演变曲线（length-scale / noise-level / log marginal likelihood）
            + σ/RMSE 校准比值曲线

  E. 实践指导
     图12: 综合 6-Panel Dashboard（RMSE/MAE/R²/Gap/σ/CV 六面板，含拐点标注）
            + Sample Requirement 预测图（目标 RMSE → 所需 n）

v3 原有 3 张图与 7 个工作表完整保留，输出目录不变。
"""

import os
import warnings
import numpy as np
import pandas as pd
import pickle
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor

warnings.filterwarnings('ignore')

print("=" * 80)
print("TOP1 GP 模型 (Gp_GP_Matern_0.5_K6_Comb17) 完整学习曲线分析程序 v4")
print("v4 新增：方差/稳定性 | 预测质量 | 收敛分析 | GP专属 | 实践指导（共12张图）")
print("=" * 80)

# ============================================================
# 路径配置
# ============================================================
PKL_FILE = (
    r"D:\博一下\第一章主动学习\高斯过程组_结果_v3.5-Extended"
    r"\top30_models\model_objects"
    r"\rank01_Gp_GP_Matern_0.5_K6_Comb17_model.pkl"
)
DATA_FILE = (
    r"D:\博一下\第一章主动学习"
    r"\Nb-Si数据库10.18-26个-4种参数-成分-性能-Si小于10的全部去掉.xlsx"
)
OUTPUT_DIR = r"D:\博二上\模型分析可视化\学习曲线分析3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 核心参数 ──────────────────────────────────────────────────
RANDOM_STATE = 2023
TEST_SIZE    = 0.2
TARGET_COL   = 'KQ'
N_ITERATIONS = 50
N_RESTARTS   = 1
TRAIN_SIZES  = [5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 130, 161]

# ============================================================
# 全局绘图风格
# ============================================================
matplotlib.rcParams['font.family']        = 'Arial'
matplotlib.rcParams['pdf.fonttype']       = 42
matplotlib.rcParams['ps.fonttype']        = 42
matplotlib.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')
SAVE_KW = dict(dpi=300, bbox_inches='tight')

# 颜色常量
C_BLUE   = '#2E86AB'
C_PINK   = '#A23B72'
C_ORANGE = '#F18F01'
C_RED    = '#C73E1D'
C_GREEN  = '#27AE60'
C_PURPLE = '#8E44AD'
C_TEAL   = '#16A085'
C_AMBER  = '#E67E22'

# ============================================================
# GP 专用工具函数
# ============================================================
def create_stratify_bins(y, n_bins=5):
    percentiles   = np.linspace(0, 100, n_bins + 1)
    bin_edges     = np.percentile(y, percentiles)
    bin_edges[0]  = -np.inf
    bin_edges[-1] =  np.inf
    return np.digitize(y, bin_edges) - 1


def gp_fit_predict(kernel, alpha, n_restarts, normalize_y,
                   X_train_raw, y_train, X_pred_raw, return_std=False):
    sc        = StandardScaler()
    X_tr_sc   = sc.fit_transform(X_train_raw)
    X_pred_sc = sc.transform(X_pred_raw)
    gp = GaussianProcessRegressor(
        kernel=kernel, alpha=alpha,
        n_restarts_optimizer=n_restarts,
        normalize_y=normalize_y,
        random_state=RANDOM_STATE,
    )
    gp.fit(X_tr_sc, y_train)
    if return_std:
        y_pred, y_std = gp.predict(X_pred_sc, return_std=True)
        return y_pred, y_std, gp
    else:
        return gp.predict(X_pred_sc), gp


def gp_fit_predict_simple(kernel, alpha, n_restarts, normalize_y,
                          X_train_raw, y_train, X_pred_raw, return_std=False):
    """原v3接口，返回值不含gp对象（向后兼容）"""
    sc        = StandardScaler()
    X_tr_sc   = sc.fit_transform(X_train_raw)
    X_pred_sc = sc.transform(X_pred_raw)
    gp = GaussianProcessRegressor(
        kernel=kernel, alpha=alpha,
        n_restarts_optimizer=n_restarts,
        normalize_y=normalize_y,
        random_state=RANDOM_STATE,
    )
    gp.fit(X_tr_sc, y_train)
    if return_std:
        y_pred, y_std = gp.predict(X_pred_sc, return_std=True)
        return y_pred, y_std
    else:
        return gp.predict(X_pred_sc)


# ============================================================
# 步骤 1: 加载模型和数据
# ============================================================
print("\n[1/8] 加载 TOP1 GP 模型和数据...")

with open(PKL_FILE, 'rb') as f:
    model_data = pickle.load(f)

gp_model_orig = model_data['model']
feature_names = model_data['features']
orig_kernel   = gp_model_orig.kernel_
orig_alpha    = gp_model_orig.get_params().get('alpha',      1e-10)
orig_norm_y   = gp_model_orig.get_params().get('normalize_y', True)

print(f"  ✓ 模型: {model_data['model_name']}")
print(f"  ✓ 特征 ({len(feature_names)}个): {feature_names}")
print(f"  ✓ 核函数: {orig_kernel}")

df       = pd.read_excel(DATA_FILE)
df_valid = df[feature_names + [TARGET_COL]].dropna()
df_valid = df_valid[df_valid[TARGET_COL] > 0].reset_index(drop=True)
X_all    = df_valid[feature_names].values
y_all    = df_valid[TARGET_COL].values

print(f"  ✓ 有效样本: {len(df_valid)},  KQ ∈ [{y_all.min():.3f}, {y_all.max():.3f}]")

# ============================================================
# 步骤 2: 固定验证集划分
# ============================================================
print("\n[2/8] 固定验证集划分（seed=2023）...")

strat   = create_stratify_bins(y_all)
idx_all = np.arange(len(y_all))
train_idx_final, val_idx_final, _, _ = train_test_split(
    idx_all, idx_all, test_size=TEST_SIZE,
    stratify=strat, random_state=RANDOM_STATE)

X_full_train = X_all[train_idx_final]
y_full_train = y_all[train_idx_final]
X_val_fixed  = X_all[val_idx_final]
y_val_fixed  = y_all[val_idx_final]

TRAIN_SIZES = [s for s in TRAIN_SIZES if s <= len(y_full_train)]
N_VAL       = len(y_val_fixed)
mean_kq     = y_val_fixed.mean()

print(f"  ✓ 训练集: {len(y_full_train)}  验证集: {N_VAL}")
print(f"  ✓ 验证集 KQ 均值: {mean_kq:.3f},  序列: {TRAIN_SIZES}")

# ============================================================
# 步骤 3: 主学习曲线循环（含 GP 超参数采集）
# ============================================================
print("\n[3/8] 学习曲线主循环（RMSE/MAE/R² + GP专属信息）...")

learning_curve_results = []
detailed_results       = []
gp_uncertainty_all     = []

# 用于后续分析的额外存储
all_val_residuals   = {}   # train_size -> list of residual arrays
all_val_preds       = {}   # train_size -> list of pred arrays
all_val_stds_per    = {}   # train_size -> list of std arrays (per-sample)
kernel_params_data  = []   # GP超参数随训练集变化

SIZES = np.array(TRAIN_SIZES)

for train_size in TRAIN_SIZES:
    print(f"  n={train_size:>4d} ...", end=" ", flush=True)

    tr_r2_l, tr_rmse_l, tr_mae_l = [], [], []
    val_r2_l, val_rmse_l, val_mae_l = [], [], []
    val_stds_l, val_stds_per_l = [], []
    val_preds_l, val_resid_l = [], []
    lml_l, ls_l, nl_l = [], [], []

    for iter_idx in range(N_ITERATIONS):
        indices    = np.random.choice(len(X_full_train), train_size, replace=False)
        X_tr_lc   = X_full_train[indices]
        y_tr_lc   = y_full_train[indices]

        try:
            val_pred, val_std, gp_obj = gp_fit_predict(
                orig_kernel, orig_alpha, N_RESTARTS, orig_norm_y,
                X_tr_lc, y_tr_lc, X_val_fixed, return_std=True)

            train_pred, _ = gp_fit_predict(
                orig_kernel, orig_alpha, N_RESTARTS, orig_norm_y,
                X_tr_lc, y_tr_lc, X_tr_lc, return_std=False)

            # 指标
            tr_r2   = r2_score(y_tr_lc,    train_pred)
            tr_rmse = np.sqrt(mean_squared_error(y_tr_lc,  train_pred))
            tr_mae  = mean_absolute_error(y_tr_lc,          train_pred)
            val_r2  = r2_score(y_val_fixed,  val_pred)
            val_rmse= np.sqrt(mean_squared_error(y_val_fixed, val_pred))
            val_mae = mean_absolute_error(y_val_fixed,        val_pred)

            tr_r2_l.append(tr_r2);   tr_rmse_l.append(tr_rmse)
            tr_mae_l.append(tr_mae)
            val_r2_l.append(val_r2); val_rmse_l.append(val_rmse)
            val_mae_l.append(val_mae)
            val_stds_l.append(val_std.mean())
            val_stds_per_l.append(val_std.copy())
            val_preds_l.append(val_pred.copy())
            val_resid_l.append((val_pred - y_val_fixed).copy())

            # GP 超参数
            try:
                lml_l.append(gp_obj.log_marginal_likelihood_value_)
                params = gp_obj.kernel_.get_params()
                # 兼容不同核结构：尝试提取 length_scale 和 noise_level
                ls_val = None
                nl_val = None
                for k, v in params.items():
                    if 'length_scale' in k and ls_val is None:
                        try:
                            ls_val = float(np.mean(v))
                        except Exception:
                            pass
                    if 'noise_level' in k and nl_val is None:
                        try:
                            nl_val = float(v)
                        except Exception:
                            pass
                if ls_val is not None:
                    ls_l.append(ls_val)
                if nl_val is not None:
                    nl_l.append(nl_val)
            except Exception:
                pass

            detailed_results.append({
                'Train_Size': train_size, 'Iteration': iter_idx + 1,
                'Train_R2': tr_r2,  'Train_RMSE': tr_rmse, 'Train_MAE': tr_mae,
                'Val_R2':  val_r2,  'Val_RMSE':  val_rmse, 'Val_MAE':  val_mae,
                'Val_GP_Mean_Std': val_std.mean(),
                'Val_GP_Max_Std':  val_std.max(),
                'Val_GP_Min_Std':  val_std.min(),
            })

        except Exception as e:
            print(f"[iter{iter_idx+1}失败:{e}]", end=" ")
            continue

    if not tr_rmse_l:
        print("全部失败，跳过")
        continue

    all_val_residuals[train_size]  = val_resid_l
    all_val_preds[train_size]      = val_preds_l
    all_val_stds_per[train_size]   = val_stds_per_l

    mean_val_std = np.mean(val_stds_l)
    std_val_std  = np.std(val_stds_l)

    learning_curve_results.append({
        'Train_Size':       train_size,
        'Train_RMSE_Mean':  np.mean(tr_rmse_l),
        'Train_RMSE_Std':   np.std(tr_rmse_l),
        'Val_RMSE_Mean':    np.mean(val_rmse_l),
        'Val_RMSE_Std':     np.std(val_rmse_l),
        'Train_MAE_Mean':   np.mean(tr_mae_l),
        'Train_MAE_Std':    np.std(tr_mae_l),
        'Val_MAE_Mean':     np.mean(val_mae_l),
        'Val_MAE_Std':      np.std(val_mae_l),
        'Train_R2_Mean':    np.mean(tr_r2_l),
        'Train_R2_Std':     np.std(tr_r2_l),
        'Val_R2_Mean':      np.mean(val_r2_l),
        'Val_R2_Std':       np.std(val_r2_l),
        'GP_Std_Mean':      mean_val_std,
        'GP_Std_Std':       std_val_std,
        'GP_CI95_Width':    mean_val_std * 1.96 * 2,
        'N_Valid_Iters':    len(tr_rmse_l),
        'Train_RMSE_CV':    np.std(tr_rmse_l) / np.mean(tr_rmse_l) * 100
                            if np.mean(tr_rmse_l) > 0 else np.nan,
        'Val_RMSE_CV':      np.std(val_rmse_l) / np.mean(val_rmse_l) * 100
                            if np.mean(val_rmse_l) > 0 else np.nan,
        'Train_MAE_CV':     np.std(tr_mae_l) / np.mean(tr_mae_l) * 100
                            if np.mean(tr_mae_l) > 0 else np.nan,
        'Val_MAE_CV':       np.std(val_mae_l) / np.mean(val_mae_l) * 100
                            if np.mean(val_mae_l) > 0 else np.nan,
        'Val_RMSE_List':    str(val_rmse_l[:5]) + '...',  # 前5仅示意
    })
    gp_uncertainty_all.append({
        'Train_Size':       train_size,
        'Mean_Uncertainty': mean_val_std,
        'Std_Uncertainty':  std_val_std,
    })
    kernel_params_data.append({
        'Train_Size':            train_size,
        'LML_Mean':              np.mean(lml_l) if lml_l else np.nan,
        'LML_Std':               np.std(lml_l)  if lml_l else np.nan,
        'LengthScale_Mean':      np.mean(ls_l)  if ls_l  else np.nan,
        'LengthScale_Std':       np.std(ls_l)   if ls_l  else np.nan,
        'NoiseLevel_Mean':       np.mean(nl_l)  if nl_l  else np.nan,
        'NoiseLevel_Std':        np.std(nl_l)   if nl_l  else np.nan,
        'Sigma_RMSE_Ratio_Mean': mean_val_std / np.mean(val_rmse_l)
                                 if np.mean(val_rmse_l) > 0 else np.nan,
    })
    print(f"ValRMSE={np.mean(val_rmse_l):.4f}±{np.std(val_rmse_l):.4f}  "
          f"R²={np.mean(val_r2_l):.3f}  σ={mean_val_std:.4f}")

# ============================================================
# 步骤 4: 数据整理
# ============================================================
print("\n[4/8] 数据整理...")

lc_df       = pd.DataFrame(learning_curve_results)
detailed_df = pd.DataFrame(detailed_results)
fig3_data   = pd.DataFrame(gp_uncertainty_all)
kp_df       = pd.DataFrame(kernel_params_data)

SIZES        = lc_df['Train_Size'].values
N_SIZES      = len(SIZES)
ci_width_arr = fig3_data['Mean_Uncertainty'].values * 1.96 * 2

# 泛化差距
analysis_rows = []
for _, row in lc_df.iterrows():
    analysis_rows.append({
        'Train_Size': row['Train_Size'],
        'RMSE_Gap':   row['Val_RMSE_Mean'] - row['Train_RMSE_Mean'],
        'MAE_Gap':    row['Val_MAE_Mean']  - row['Train_MAE_Mean'],
        'R2_Gap':     row['Train_R2_Mean'] - row['Val_R2_Mean'],
        'Train_RMSE_CV': row['Train_RMSE_CV'],
        'Val_RMSE_CV':   row['Val_RMSE_CV'],
    })
analysis_df = pd.DataFrame(analysis_rows)

rmse_improv = ((lc_df['Val_RMSE_Mean'].iloc[0] - lc_df['Val_RMSE_Mean'].iloc[-1]) /
               lc_df['Val_RMSE_Mean'].iloc[0] * 100)
mae_improv  = ((lc_df['Val_MAE_Mean'].iloc[0]  - lc_df['Val_MAE_Mean'].iloc[-1]) /
               lc_df['Val_MAE_Mean'].iloc[0]  * 100)
r2_change   = lc_df['Val_R2_Mean'].iloc[-1] - lc_df['Val_R2_Mean'].iloc[0]
r2_init, r2_final = lc_df['Val_R2_Mean'].iloc[0], lc_df['Val_R2_Mean'].iloc[-1]
conv_start  = SIZES[int(len(SIZES) * 0.7)]

# PI 校准：计算每个训练集大小下 95% PI 的实际覆盖率
calibration_rows  = []
picp_mpiw_rows    = []
coverage_levels   = np.arange(0.05, 1.0, 0.05)

for train_size in SIZES:
    if train_size not in all_val_stds_per:
        continue
    preds_list = all_val_preds[train_size]
    stds_list  = all_val_stds_per[train_size]

    picp_list, mpiw_list = [], []
    for pred_arr, std_arr in zip(preds_list, stds_list):
        lower = pred_arr - 1.96 * std_arr
        upper = pred_arr + 1.96 * std_arr
        covered = np.mean((y_val_fixed >= lower) & (y_val_fixed <= upper))
        pi_width = np.mean(upper - lower)
        picp_list.append(covered)
        mpiw_list.append(pi_width)

    picp_mpiw_rows.append({
        'Train_Size':  train_size,
        'PICP_Mean':   np.mean(picp_list),
        'PICP_Std':    np.std(picp_list),
        'MPIW_Mean':   np.mean(mpiw_list),
        'MPIW_Std':    np.std(mpiw_list),
    })

    # 对所有iterations拼接residuals，计算不同置信水平的覆盖率
    all_preds_flat = np.concatenate(preds_list)
    all_stds_flat  = np.concatenate(stds_list)
    y_rep = np.tile(y_val_fixed, len(preds_list))

    cov_obs = []
    for z in stats.norm.ppf((1 + coverage_levels) / 2):
        lo = all_preds_flat - z * all_stds_flat
        hi = all_preds_flat + z * all_stds_flat
        cov_obs.append(np.mean((y_rep >= lo) & (y_rep <= hi)))
    calibration_rows.append({
        'Train_Size':      train_size,
        'Coverage_Levels': list(coverage_levels),
        'Coverage_Obs':    cov_obs,
    })

picp_df = pd.DataFrame(picp_mpiw_rows)

# 幂律拟合数据
def power_law(n, a, b):
    return a * np.power(n, b)

val_rmse_arr = lc_df['Val_RMSE_Mean'].values
try:
    popt_val, _ = curve_fit(power_law, SIZES, val_rmse_arr,
                            p0=[5.0, -0.3], maxfev=5000)
    a_val, b_val = popt_val
    pl_fit_ok = True
except Exception:
    pl_fit_ok = False
    a_val, b_val = np.nan, np.nan

# 边际收益递减
marginal_rmse = np.diff(val_rmse_arr)
marginal_n    = np.diff(SIZES.astype(float))
marginal_rate = marginal_rmse / marginal_n  # ΔRMSE/ΔN，负值=改善

# Bootstrap CI for learning curve (1000次重采样)
print("    计算 Bootstrap CI...")
np.random.seed(RANDOM_STATE)
N_BOOT = 1000
boot_val_rmse = np.zeros((N_BOOT, N_SIZES))
for b_idx in range(N_BOOT):
    for s_idx, ts in enumerate(SIZES):
        if ts not in all_val_residuals:
            boot_val_rmse[b_idx, s_idx] = np.nan
            continue
        rmse_list = [np.sqrt(np.mean(r**2)) for r in all_val_residuals[ts]]
        if not rmse_list:
            boot_val_rmse[b_idx, s_idx] = np.nan
            continue
        boot_sample = np.random.choice(rmse_list, len(rmse_list), replace=True)
        boot_val_rmse[b_idx, s_idx] = np.mean(boot_sample)

boot_ci_lo = np.nanpercentile(boot_val_rmse, 2.5,  axis=0)
boot_ci_hi = np.nanpercentile(boot_val_rmse, 97.5, axis=0)

# Area between curves
rmse_gap_arr = analysis_df['RMSE_Gap'].values
abc_total    = np.trapz(np.maximum(rmse_gap_arr, 0), SIZES)

# Sample size requirement prediction
if pl_fit_ok and b_val < 0:
    target_rmses   = np.linspace(val_rmse_arr[-1] * 0.5,
                                 val_rmse_arr[-1] * 1.5, 100)
    required_ns    = np.power(target_rmses / a_val, 1.0 / b_val)
    # 需要满足 n > 0
    mask_pos = required_ns > 0
    target_rmses = target_rmses[mask_pos]
    required_ns  = required_ns[mask_pos]
else:
    target_rmses, required_ns = None, None

# 肘点检测（二阶差分法）
second_diff   = np.diff(np.diff(val_rmse_arr))
elbow_idx_raw = np.argmax(np.abs(second_diff)) + 1  # +1 因为两次diff偏移
elbow_idx     = min(elbow_idx_raw, len(SIZES) - 1)
elbow_size    = SIZES[elbow_idx]
elbow_rmse    = val_rmse_arr[elbow_idx]

print(f"    ✓ 幂律拟合: RMSE = {a_val:.4f}·n^{b_val:.4f}  (拟合{'成功' if pl_fit_ok else '失败'})")
print(f"    ✓ 最优拐点: n={elbow_size}  Val RMSE={elbow_rmse:.4f}")
print(f"    ✓ 泛化代价积分 (Area Between Curves): {abc_total:.4f}")

# ============================================================
# 步骤 5: 保存完整 Excel（17个工作表）
# ============================================================
print("\n[5/8] 保存 Excel（17个工作表）...")

excel_path = os.path.join(OUTPUT_DIR, 'GP_Learning_Curve_v4_Complete_Data.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # ── 原 v3 工作表 ──────────────────────────────────────────
    lc_df.to_excel(      writer, sheet_name='Summary',             index=False)
    detailed_df.to_excel(writer, sheet_name='Detailed_All_Iters',  index=False)
    analysis_df.to_excel(writer, sheet_name='Analysis_Gaps',       index=False)
    lc_df[[
        'Train_Size',
        'Train_RMSE_Mean','Train_RMSE_Std','Val_RMSE_Mean','Val_RMSE_Std',
        'Train_MAE_Mean', 'Train_MAE_Std', 'Val_MAE_Mean', 'Val_MAE_Std',
        'Train_R2_Mean',  'Train_R2_Std',  'Val_R2_Mean',  'Val_R2_Std',
    ]].to_excel(writer, sheet_name='Fig1_Learning_Curves',     index=False)
    analysis_df[['Train_Size','RMSE_Gap','MAE_Gap','R2_Gap']].to_excel(
        writer, sheet_name='Fig2_Generalization_Gaps', index=False)
    fig3_data.to_excel(  writer, sheet_name='Fig3_GP_Uncertainty',  index=False)

    # ── v4 新增工作表 ─────────────────────────────────────────
    # CV 稳定性
    cv_df = lc_df[['Train_Size','Train_RMSE_CV','Val_RMSE_CV',
                   'Train_MAE_CV','Val_MAE_CV']].copy()
    cv_df.to_excel(writer, sheet_name='FigA_CV_Stability', index=False)

    # PICP-MPIW
    picp_df.to_excel(writer, sheet_name='FigB_PICP_MPIW', index=False)

    # Calibration
    cal_flat = []
    for row in calibration_rows:
        for cl, co in zip(row['Coverage_Levels'], row['Coverage_Obs']):
            cal_flat.append({'Train_Size': row['Train_Size'],
                             'Expected_Coverage': cl,
                             'Observed_Coverage': co})
    pd.DataFrame(cal_flat).to_excel(
        writer, sheet_name='FigB_Calibration', index=False)

    # 幂律 + 边际收益
    pl_df = pd.DataFrame({
        'Train_Size':      SIZES,
        'Val_RMSE_Mean':   val_rmse_arr,
        'PowerLaw_Fit':    power_law(SIZES, a_val, b_val) if pl_fit_ok else [np.nan]*N_SIZES,
        'Marginal_ΔRMSE':  np.append([np.nan], marginal_rmse),
        'Marginal_ΔN':     np.append([np.nan], marginal_n),
        'Marginal_Rate':   np.append([np.nan], marginal_rate),
    })
    pl_df.to_excel(writer, sheet_name='FigC_PowerLaw_Marginal', index=False)

    # Bootstrap CI
    boot_df = pd.DataFrame({
        'Train_Size':     SIZES,
        'Val_RMSE_Mean':  val_rmse_arr,
        'Boot_CI_Lo_2.5': boot_ci_lo,
        'Boot_CI_Hi_97.5':boot_ci_hi,
        'ABC_Total':      [abc_total] + [np.nan] * (N_SIZES - 1),
    })
    boot_df.to_excel(writer, sheet_name='FigC_Bootstrap_CI', index=False)

    # 核超参数
    kp_df.to_excel(writer, sheet_name='FigD_Kernel_Params', index=False)

    # Sample Requirement
    if target_rmses is not None:
        pd.DataFrame({
            'Target_Val_RMSE':  target_rmses,
            'Required_N':       required_ns,
        }).to_excel(writer, sheet_name='FigE_Sample_Requirement', index=False)

    # 配置记录
    pd.DataFrame({
        'Parameter': [
            'Model','Features','Total_Samples','Val_Split','Seed','Train_Sizes',
            'N_Iterations','GP_Kernel','GP_Alpha','n_restarts',
            'PowerLaw_a','PowerLaw_b','Elbow_Size','Elbow_RMSE',
            'ABC_Total','v4_New_Figures',
        ],
        'Value': [
            'Gp_GP_Matern_0.5_K6_Comb17', ', '.join(feature_names),
            len(y_all), f'{TEST_SIZE*100:.0f}%', RANDOM_STATE, str(list(SIZES)),
            N_ITERATIONS, str(orig_kernel), orig_alpha, N_RESTARTS,
            f'{a_val:.6f}', f'{b_val:.6f}', elbow_size, f'{elbow_rmse:.6f}',
            f'{abc_total:.4f}',
            'Fig4(Box+Violin),Fig5(CV),Fig6(Residual),Fig7(Calibration),'
            'Fig8(PICP-MPIW),Fig9(PowerLaw+Marginal),Fig10(Bootstrap+ABC),'
            'Fig11(KernelParams+σ/RMSE),Fig12(Dashboard+SampleReq)',
        ],
    }).to_excel(writer, sheet_name='Configuration_v4', index=False)

print(f"  ✓ Excel 已保存（17个工作表）: {os.path.basename(excel_path)}")

# ============================================================
# 步骤 6: 原 v3 三张图（完整保留）
# ============================================================
print("\n[6/8] 生成原 v3 三张图（Fig1–Fig3）...")

# ── 图1: RMSE / MAE / R² 学习曲线 ───────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 18))

for ax, (tr_col, tr_std, va_col, va_std, metric, tr_c, va_c) in zip(axes, [
    ('Train_RMSE_Mean','Train_RMSE_Std','Val_RMSE_Mean','Val_RMSE_Std','RMSE',C_BLUE,C_PINK),
    ('Train_MAE_Mean', 'Train_MAE_Std', 'Val_MAE_Mean', 'Val_MAE_Std', 'MAE', C_ORANGE, C_RED),
    ('Train_R2_Mean',  'Train_R2_Std',  'Val_R2_Mean',  'Val_R2_Std',  'R²',  C_GREEN,  C_PURPLE),
]):
    ax.plot(SIZES, lc_df[tr_col], 'o-', color=tr_c, lw=3, ms=9,
            label=f'Training {metric}', zorder=3)
    ax.fill_between(SIZES, lc_df[tr_col]-lc_df[tr_std],
                    lc_df[tr_col]+lc_df[tr_std],
                    alpha=0.20, color=tr_c)
    ax.plot(SIZES, lc_df[va_col], 's-', color=va_c, lw=3, ms=9,
            label=f'Validation {metric}', zorder=3)
    ax.fill_between(SIZES, lc_df[va_col]-lc_df[va_std],
                    lc_df[va_col]+lc_df[va_std],
                    alpha=0.20, color=va_c)

    z_tr = np.polyfit(SIZES, lc_df[tr_col], 1)
    z_va = np.polyfit(SIZES, lc_df[va_col], 1)
    ax.plot(SIZES, np.poly1d(z_tr)(SIZES), '--', color=tr_c, lw=2, alpha=0.7,
            label=f'Train trend (slope={z_tr[0]:.4f})')
    ax.plot(SIZES, np.poly1d(z_va)(SIZES), '--', color=va_c, lw=2, alpha=0.7,
            label=f'Val trend (slope={z_va[0]:.4f})')

    if metric == 'R²':
        y_min_r2 = min(lc_df[va_col].min() - lc_df[va_std].max(), -0.1)
        if y_min_r2 < 0:
            ax.axhspan(y_min_r2 - 0.05, 0, alpha=0.08, color='red', zorder=0)
        ax.axhline(y=0, color='red', ls='-', lw=2.0, alpha=0.8,
                   label='R²=0 (no better than mean)', zorder=4)
        ax.axhline(y=1, color='black', ls=':', lw=1.5, alpha=0.4,
                   label='R²=1 (perfect fit)')
        val_final = lc_df[va_col].iloc[-1]
        sign_str  = '+' if r2_change >= 0 else ''
        ax.text(0.98, 0.05,
                f'Val R² Change:\n{sign_str}{r2_change:.4f}\n'
                f'({r2_init:.4f} → {r2_final:.4f})',
                transform=ax.transAxes, fontsize=12, fontweight='bold',
                va='bottom', ha='right',
                bbox=dict(boxstyle='round', fc='#E6E6FA', alpha=0.85,
                          ec='purple', lw=2))
    else:
        val_final = lc_df[va_col].iloc[-1]
        improv = ((lc_df[va_col].iloc[0] - val_final) /
                  lc_df[va_col].iloc[0] * 100)
        fc = '#90EE90' if metric == 'RMSE' else '#FFD700'
        ec = 'green'  if metric == 'RMSE' else 'orange'
        ax.text(0.98, 0.95,
                f'Val {metric} Improvement:\n{improv:.1f}%',
                transform=ax.transAxes, fontsize=13, fontweight='bold',
                va='top', ha='right',
                bbox=dict(boxstyle='round', fc=fc, alpha=0.85, ec=ec, lw=2))

    offset = (lc_df[va_col].max() - lc_df[va_col].min()) * 0.05
    ax.annotate(f'Final: {val_final:.4f}',
                xy=(SIZES[-1], val_final),
                xytext=(SIZES[-1] - 40, val_final + offset),
                fontsize=11, fontweight='bold', color=va_c,
                bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=va_c, lw=2),
                arrowprops=dict(arrowstyle='->', color=va_c, lw=2))

    ax.set_xlabel('Training Set Size (samples)', fontsize=13, fontweight='bold')
    ax.set_ylabel(metric, fontsize=13, fontweight='bold')
    ax.set_title(
        f'Learning Curve: {metric}  |  GP Matérn-0.5  |  '
        f'{N_ITERATIONS} iterations/size  |  seed={RANDOM_STATE}',
        fontsize=13, fontweight='bold')
    ax.legend(fontsize=11, framealpha=0.95)
    ax.set_xticks(SIZES)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, ls='--')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '1_learning_curve_RMSE_MAE_R2_v4.png'), **SAVE_KW)
plt.close()
print("  ✓ 图1 已保存")

# ── 图2: 泛化差距 ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
rmse_gap_arr = analysis_df['RMSE_Gap'].values
mae_gap_arr  = analysis_df['MAE_Gap'].values
r2_gap_arr   = analysis_df['R2_Gap'].values

for ax, gap_arr, metric, color, risk_base in [
    (axes[0], rmse_gap_arr, 'RMSE', C_ORANGE, lc_df['Val_RMSE_Mean'].max()),
    (axes[1], mae_gap_arr,  'MAE',  C_RED,    lc_df['Val_MAE_Mean'].max()),
    (axes[2], r2_gap_arr,   'R²',  C_PURPLE, None),
]:
    if risk_base:
        ax.axhspan(risk_base*0.50, max(gap_arr.max(), risk_base)*1.15,
                   alpha=0.12, color='red',    zorder=0, label='High Risk')
        ax.axhspan(risk_base*0.25, risk_base*0.50,
                   alpha=0.12, color='orange', zorder=0, label='Medium Risk')
        ax.axhspan(0, risk_base*0.25, alpha=0.12, color='green',  zorder=0,
                   label='Low Risk')
    else:
        ax.axhspan(0.30, max(gap_arr.max(), 0.35)*1.1, alpha=0.12,
                   color='red',    zorder=0, label='High Risk (>0.30)')
        ax.axhspan(0.15, 0.30, alpha=0.12, color='orange', zorder=0,
                   label='Medium Risk')
        ax.axhspan(0, 0.15, alpha=0.12, color='green', zorder=0,
                   label='Low Risk (<0.15)')

    marker = 'o-' if metric == 'RMSE' else ('s-' if metric == 'MAE' else 'd-')
    ax.plot(SIZES, gap_arr, marker, color=color, lw=3, ms=9,
            label=f'{metric} Gap', zorder=3)
    ax.axhline(0, color='black', lw=2, alpha=0.5)
    z = np.polyfit(SIZES, gap_arr, 1)
    ax.plot(SIZES, np.poly1d(z)(SIZES), '--', color=color, lw=2, alpha=0.7,
            label=f'Trend (slope={z[0]:.4f})')
    for i, (s, g) in enumerate(zip(SIZES, gap_arr)):
        if i % 2 == 0:
            ax.text(s, g + abs(gap_arr).max() * 0.03, f'{g:.3f}',
                    ha='center', fontsize=9, fontweight='bold')
    ax.axvspan(conv_start, SIZES[-1], alpha=0.08, color='blue')
    ax.text((conv_start + SIZES[-1]) / 2, abs(gap_arr).max() * 0.06,
            'Stabilizing', ha='center', fontsize=9,
            fontweight='bold', color='blue', style='italic')
    label_y = 'Val − Train' if metric != 'R²' else 'Train − Val'
    ax.set_xlabel('Training Set Size', fontsize=12, fontweight='bold')
    ax.set_ylabel(f'{metric} Gap ({label_y})', fontsize=12, fontweight='bold')
    ax.set_title(f'Generalization Gap: {metric}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xticks(SIZES)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, ls='--')

plt.suptitle(f'GP Matérn-0.5 — Generalization Gap Analysis (v4)  |  seed={RANDOM_STATE}',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '2_generalization_gaps_v4.png'), **SAVE_KW)
plt.close()
print("  ✓ 图2 已保存")

# ── 图3: GP 置信区间 ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
ax = axes[0]
ax.plot(fig3_data['Train_Size'], fig3_data['Mean_Uncertainty'],
        'o-', color='#9B59B6', lw=3, ms=9, label='Mean GP σ')
ax.fill_between(fig3_data['Train_Size'],
                fig3_data['Mean_Uncertainty'] - fig3_data['Std_Uncertainty'],
                fig3_data['Mean_Uncertainty'] + fig3_data['Std_Uncertainty'],
                alpha=0.20, color='#9B59B6')
z_u = np.polyfit(fig3_data['Train_Size'], fig3_data['Mean_Uncertainty'], 1)
ax.plot(fig3_data['Train_Size'], np.poly1d(z_u)(fig3_data['Train_Size']),
        '--', color='#9B59B6', lw=2, alpha=0.7,
        label=f'Trend (slope={z_u[0]:.5f})')
kq_range = y_val_fixed.max() - y_val_fixed.min()
ax.axhline(y=kq_range*0.05, color='orange', ls='--', lw=2, alpha=0.8,
           label=f'5% KQ range threshold')
ax.set_xlabel('Training Set Size', fontsize=13, fontweight='bold')
ax.set_ylabel('GP Prediction Std (σ)', fontsize=13, fontweight='bold')
ax.set_title('GP Native Uncertainty vs Sample Size\n(Bayesian Posterior Std)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(fig3_data['Train_Size'])
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

ax = axes[1]
bars = ax.bar(range(N_SIZES), ci_width_arr, color=C_TEAL,
              alpha=0.75, edgecolor='black', lw=1.5)
ax.set_xticks(range(N_SIZES))
ax.set_xticklabels([f'n={s}' for s in SIZES], rotation=45, ha='right')
ax.set_ylabel('95% CI Width', fontsize=13, fontweight='bold')
initial_ci_half = ci_width_arr[0]/2
final_ci_half   = ci_width_arr[-1]/2
reduction_pct   = (1 - final_ci_half / initial_ci_half) * 100
arrow = FancyArrowPatch((0, ci_width_arr[0]), (N_SIZES-1, ci_width_arr[-1]),
                        arrowstyle='->', mutation_scale=30, lw=3,
                        color='red', zorder=5)
ax.add_patch(arrow)
ax.text(N_SIZES/2-1,
        (ci_width_arr[0]+ci_width_arr[-1])/2 + ci_width_arr.max()*0.05,
        f'{reduction_pct:.1f}% Reduction\nin GP Uncertainty',
        ha='center', fontsize=13, fontweight='bold', color='red',
        bbox=dict(boxstyle='round,pad=0.7', fc='yellow', alpha=0.90,
                  ec='red', lw=2.5))
for bar, val in zip(bars, ci_width_arr):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + ci_width_arr.max()*0.01,
            f'±{val/2:.3f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold')
ax.set_title(f'GP Prediction Confidence Intervals  |  n={SIZES[-1]}: ±{final_ci_half:.4f} (95% CI)',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'GP Matérn-0.5 — GP Bayesian Uncertainty vs Sample Size (v4)  |  seed={RANDOM_STATE}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '3_GP_confidence_interval_v4.png'), **SAVE_KW)
plt.close()
print("  ✓ 图3 已保存")

# ============================================================
# 步骤 7: v4 新增八张图（Fig4–Fig12）
# ============================================================
print("\n[7/8] 生成 v4 新增可视化（Fig4–Fig12）...")

# ── 图4: Box-whisker + Violin 双联图 ─────────────────────────
print("  绘制 图4: Box-whisker + Violin...")

# 从 detailed_df 提取每个 train_size 的 Val_RMSE 列表
fig, axes = plt.subplots(3, 2, figsize=(22, 18))
metrics_box = [
    ('Val_RMSE', 'Val RMSE', C_PINK),
    ('Val_MAE',  'Val MAE',  C_RED),
    ('Val_R2',   'Val R²',  C_PURPLE),
]
for row_idx, (col, label, color) in enumerate(metrics_box):
    data_by_size = [
        detailed_df[detailed_df['Train_Size'] == ts][col].values
        for ts in SIZES
    ]
    labels = [f'n={s}' for s in SIZES]

    # 左列：Box-whisker
    ax_box = axes[row_idx][0]
    bp = ax_box.boxplot(data_by_size, labels=labels, patch_artist=True,
                        medianprops=dict(color='black', lw=2.5),
                        whiskerprops=dict(color=color, lw=1.5),
                        capprops=dict(color=color, lw=2),
                        flierprops=dict(marker='o', markerfacecolor=color,
                                        markersize=4, alpha=0.5))
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.45)
    ax_box.set_xlabel('Training Set Size', fontsize=12, fontweight='bold')
    ax_box.set_ylabel(label, fontsize=12, fontweight='bold')
    ax_box.set_title(f'Box-Whisker: {label} Distribution  |  {N_ITERATIONS} iters/size',
                     fontsize=13, fontweight='bold')
    ax_box.tick_params(axis='x', rotation=45)
    ax_box.grid(True, alpha=0.3, ls='--', axis='y')

    # 右列：Violin
    ax_vio = axes[row_idx][1]
    vp = ax_vio.violinplot(data_by_size, positions=range(len(SIZES)),
                           showmeans=True, showmedians=True)
    for body in vp['bodies']:
        body.set_facecolor(color)
        body.set_alpha(0.40)
    for part in ['cmeans', 'cmedians', 'cbars', 'cmins', 'cmaxes']:
        if part in vp:
            vp[part].set_color(color)
            vp[part].set_linewidth(2)
    ax_vio.set_xticks(range(len(SIZES)))
    ax_vio.set_xticklabels(labels, rotation=45)
    ax_vio.set_xlabel('Training Set Size', fontsize=12, fontweight='bold')
    ax_vio.set_ylabel(label, fontsize=12, fontweight='bold')
    ax_vio.set_title(f'Violin Plot: {label} Full Distribution',
                     fontsize=13, fontweight='bold')
    ax_vio.grid(True, alpha=0.3, ls='--', axis='y')

plt.suptitle(
    f'GP Matérn-0.5 — Variance & Stability Analysis (v4)  |  '
    f'{N_ITERATIONS} iterations/size  |  seed={RANDOM_STATE}',
    fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '4_box_violin_variance_v4.png'), **SAVE_KW)
plt.close()
print("    ✓ 图4 已保存")

# ── 图5: 变异系数 CV(%) 曲线 ──────────────────────────────────
print("  绘制 图5: CV(%) 曲线...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (tr_cv, va_cv, label, tr_c, va_c) in zip(axes, [
    ('Train_RMSE_CV', 'Val_RMSE_CV', 'RMSE', C_BLUE,   C_PINK),
    ('Train_MAE_CV',  'Val_MAE_CV',  'MAE',  C_ORANGE,  C_RED),
]):
    ax.plot(SIZES, lc_df[tr_cv], 'o-', color=tr_c, lw=3, ms=9,
            label=f'Training {label} CV%', zorder=3)
    ax.plot(SIZES, lc_df[va_cv], 's-', color=va_c, lw=3, ms=9,
            label=f'Validation {label} CV%', zorder=3)
    ax.axhline(y=10, color='green',  ls='--', lw=2, alpha=0.8,
               label='10% threshold (stable)')
    ax.axhline(y=30, color='orange', ls='--', lw=2, alpha=0.8,
               label='30% threshold (unstable)')
    ax.fill_between(SIZES, 0, 10, alpha=0.06, color='green')
    ax.fill_between(SIZES, 10, 30, alpha=0.06, color='orange')
    ax.fill_between(SIZES, 30,
                    max(lc_df[tr_cv].max(), lc_df[va_cv].max())*1.1,
                    alpha=0.06, color='red')
    for s, tcv, vcv in zip(SIZES, lc_df[tr_cv], lc_df[va_cv]):
        ax.text(s, vcv + 0.5, f'{vcv:.1f}', ha='center', fontsize=8,
                color=va_c, fontweight='bold')
    ax.set_xlabel('Training Set Size', fontsize=12, fontweight='bold')
    ax.set_ylabel('CV = Std/Mean × 100 (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Coefficient of Variation: {label}  |  '
                 f'Prediction Stability Diagnosis',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_xticks(SIZES)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, ls='--')

plt.suptitle(
    f'GP Matérn-0.5 — CV(%) Stability Analysis (v4)  |  seed={RANDOM_STATE}',
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '5_CV_stability_v4.png'), **SAVE_KW)
plt.close()
print("    ✓ 图5 已保存")

# ── 图6: 残差分布演变图 ───────────────────────────────────────
print("  绘制 图6: 残差分布演变...")

rep_sizes = []
for target in [SIZES[0], SIZES[len(SIZES)//4],
               SIZES[len(SIZES)//2], SIZES[-1]]:
    closest = SIZES[np.argmin(np.abs(SIZES - target))]
    if closest not in rep_sizes:
        rep_sizes.append(closest)
rep_sizes = rep_sizes[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes_flat = axes.flatten()

for ax, ts in zip(axes_flat, rep_sizes):
    if ts not in all_val_residuals:
        ax.set_visible(False)
        continue
    all_resid = np.concatenate(all_val_residuals[ts])
    mean_r = np.mean(all_resid)
    std_r  = np.std(all_resid)
    sk     = stats.skew(all_resid)
    ku     = stats.kurtosis(all_resid)

    ax.hist(all_resid, bins=40, density=True,
            color=C_BLUE, alpha=0.45, edgecolor='white', lw=0.5,
            label='Residual distribution')
    x_range = np.linspace(all_resid.min(), all_resid.max(), 300)
    kde = stats.gaussian_kde(all_resid)
    ax.plot(x_range, kde(x_range), color=C_PINK, lw=2.5, label='KDE')
    # 正态参考
    ax.plot(x_range, stats.norm.pdf(x_range, mean_r, std_r),
            color=C_GREEN, lw=2, ls='--', label='Normal reference')
    ax.axvline(0, color='red', lw=2, ls='-', label='Zero bias')
    ax.axvline(mean_r, color=C_AMBER, lw=2, ls='--',
               label=f'Mean bias={mean_r:.3f}')

    textstr = (f'n={ts}  Residuals×{len(all_val_residuals[ts])} iters\n'
               f'Mean={mean_r:.3f}  Std={std_r:.3f}\n'
               f'Skewness={sk:.3f}  Kurtosis={ku:.3f}')
    ax.text(0.97, 0.96, textstr, transform=ax.transAxes,
            fontsize=10, va='top', ha='right',
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9,
                      ec='orange', lw=1.5))
    ax.set_xlabel('Prediction − True (MPa·m¹/²)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Density', fontsize=12, fontweight='bold')
    ax.set_title(f'Residual Distribution  |  n_train={ts}',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3, ls='--')

plt.suptitle(
    f'GP Matérn-0.5 — Residual Distribution Evolution (v4)  |  seed={RANDOM_STATE}',
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '6_residual_distribution_v4.png'), **SAVE_KW)
plt.close()
print("    ✓ 图6 已保存")

# ── 图7: GP 不确定性校准曲线 ─────────────────────────────────
print("  绘制 图7: Calibration curve...")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左图：Expected vs Observed coverage（代表性训练集大小）
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Perfect calibration')
colors_cal = plt.cm.viridis(np.linspace(0.1, 0.9, len(calibration_rows)))

for i, row in enumerate(calibration_rows):
    ax.plot(row['Coverage_Levels'], row['Coverage_Obs'],
            '-o', color=colors_cal[i], lw=2, ms=5, alpha=0.85,
            label=f'n={row["Train_Size"]}')

ax.fill_between([0, 1], [0, 1], [0.05, 1.05],
                alpha=0.05, color='green', label='±5% calibration band')
ax.fill_between([0, 1], [0, 1], [-0.05, 0.95],
                alpha=0.05, color='green')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Expected Coverage (nominal CI)', fontsize=12, fontweight='bold')
ax.set_ylabel('Observed Coverage (actual hit rate)', fontsize=12, fontweight='bold')
ax.set_title('GP Uncertainty Calibration Curve\n'
             '(Above diagonal = over-confident; Below = conservative)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, ncol=2, framealpha=0.9)
ax.grid(True, alpha=0.3, ls='--')

# 右图：95% PI actual coverage rate vs train size
ax = axes[1]
if not picp_df.empty:
    ax.plot(picp_df['Train_Size'], picp_df['PICP_Mean'] * 100,
            'o-', color=C_PURPLE, lw=3, ms=9, label='Actual 95% PI coverage rate')
    ax.fill_between(picp_df['Train_Size'],
                    (picp_df['PICP_Mean'] - picp_df['PICP_Std']) * 100,
                    (picp_df['PICP_Mean'] + picp_df['PICP_Std']) * 100,
                    alpha=0.2, color=C_PURPLE)
    ax.axhline(y=95, color='red', ls='--', lw=2.5,
               label='Nominal 95% target')
    ax.axhspan(90, 100, alpha=0.08, color='green',
               label='Acceptable range (90–100%)')

    for ts, picp in zip(picp_df['Train_Size'], picp_df['PICP_Mean']):
        ax.text(ts, picp * 100 + 0.5, f'{picp*100:.1f}%',
                ha='center', fontsize=9, fontweight='bold', color=C_PURPLE)

ax.set_xlabel('Training Set Size', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual 95% PI Coverage Rate (%)', fontsize=12, fontweight='bold')
ax.set_title('95% Prediction Interval Coverage\nvs Training Size',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11, framealpha=0.9)
ax.set_xticks(SIZES)
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, ls='--')

plt.suptitle(
    f'GP Matérn-0.5 — Uncertainty Calibration Analysis (v4)  |  seed={RANDOM_STATE}',
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '7_calibration_PI_coverage_v4.png'), **SAVE_KW)
plt.close()
print("    ✓ 图7 已保存")

# ── 图8: PICP-MPIW 联合图 ────────────────────────────────────
print("  绘制 图8: PICP-MPIW...")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
if not picp_df.empty:
    sc = ax.scatter(picp_df['MPIW_Mean'], picp_df['PICP_Mean'] * 100,
                    c=picp_df['Train_Size'], cmap='viridis',
                    s=120, zorder=3, edgecolors='black', lw=1.5)
    plt.colorbar(sc, ax=ax, label='Training Size')
    for _, row in picp_df.iterrows():
        ax.annotate(f'n={int(row["Train_Size"])}',
                    (row['MPIW_Mean'], row['PICP_Mean']*100),
                    textcoords='offset points', xytext=(6, 4), fontsize=9)
    ax.axhline(y=95, color='red', ls='--', lw=2, label='95% nominal')
    ax.axhspan(90, 100, alpha=0.08, color='green', label='Acceptable zone')
    ax.set_xlabel('Mean Prediction Interval Width (MPIW)', fontsize=12,
                  fontweight='bold')
    ax.set_ylabel('PI Coverage Probability (PICP %)', fontsize=12,
                  fontweight='bold')
    ax.set_title('PICP-MPIW Tradeoff\n(Ideal: high PICP near 95% + narrow MPIW)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, ls='--')

ax = axes[1]
if not picp_df.empty:
    ax2_twin = ax.twinx()
    ax.bar(range(N_SIZES), picp_df['MPIW_Mean'],
           color=C_TEAL, alpha=0.55, edgecolor='black', lw=1,
           label='Mean PI Width (MPIW)')
    ax2_twin.plot(range(N_SIZES), picp_df['PICP_Mean'] * 100,
                  'o-', color=C_RED, lw=3, ms=9,
                  label='PICP (%)')
    ax2_twin.axhline(y=95, color='red', ls='--', lw=2)
    ax2_twin.set_ylabel('PICP (%)', fontsize=12, fontweight='bold', color=C_RED)
    ax.set_xticks(range(N_SIZES))
    ax.set_xticklabels([f'n={s}' for s in SIZES], rotation=45, ha='right')
    ax.set_xlabel('Training Set Size', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean PI Width (MPIW)', fontsize=12, fontweight='bold',
                  color=C_TEAL)
    ax.set_title('MPIW & PICP vs Training Size', fontsize=13, fontweight='bold')
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2_twin.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labs1 + labs2, fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    f'GP Matérn-0.5 — PICP-MPIW Uncertainty Quality (v4)  |  seed={RANDOM_STATE}',
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '8_PICP_MPIW_v4.png'), **SAVE_KW)
plt.close()
print("    ✓ 图8 已保存")

# ── 图9: 幂律拟合 + 边际收益递减 ─────────────────────────────
print("  绘制 图9: 幂律拟合 + 边际收益递减...")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左：幂律拟合
ax = axes[0]
ax.plot(SIZES, val_rmse_arr, 'o-', color=C_PINK, lw=3, ms=10,
        label='Val RMSE (observed)', zorder=3)

if pl_fit_ok:
    n_dense = np.linspace(SIZES[0], max(SIZES[-1], 250), 500)
    ax.plot(n_dense, power_law(n_dense, a_val, b_val), '--',
            color=C_PURPLE, lw=2.5, alpha=0.9,
            label=f'Power-law fit: RMSE = {a_val:.4f}·n^{b_val:.4f}')
    # 拐点
    ax.axvline(x=elbow_size, color='orange', ls=':', lw=2.5,
               label=f'Elbow point: n={elbow_size}')
    ax.scatter([elbow_size], [elbow_rmse], color='orange', s=200,
               zorder=5, edgecolors='black', lw=2)
    # 预测延伸区域
    ax.fill_betweenx([0, val_rmse_arr.max()*1.1],
                     SIZES[-1], max(SIZES[-1], 250),
                     alpha=0.05, color='green',
                     label=f'Extrapolation zone (n>{SIZES[-1]})')
    ax.text(0.97, 0.95,
            f'Power-law: RMSE = {a_val:.4f}·n^{b_val:.4f}\n'
            f'Convergence rate b = {b_val:.4f}\n'
            f'(|b| larger → faster convergence)\n'
            f'Elbow: n={elbow_size}, RMSE={elbow_rmse:.4f}',
            transform=ax.transAxes, fontsize=11, va='top', ha='right',
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9,
                      ec='orange', lw=1.5))

ax.set_xlabel('Training Set Size (n)', fontsize=12, fontweight='bold')
ax.set_ylabel('Val RMSE', fontsize=12, fontweight='bold')
ax.set_title('Power-Law Convergence Fit\n'
             'RMSE = a·nᵇ  (b<0 → decreasing)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, framealpha=0.9)
ax.set_xticks(SIZES)
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, ls='--')

# 右：边际收益递减
ax = axes[1]
mid_sizes = (SIZES[:-1] + SIZES[1:]) / 2
bar_colors = [C_GREEN if v < 0 else C_RED for v in marginal_rate]
bars = ax.bar(range(len(mid_sizes)), marginal_rate,
              color=bar_colors, alpha=0.70, edgecolor='black', lw=1)
ax.axhline(0, color='black', lw=2)
ax.axhline(y=-0.005, color='orange', ls='--', lw=2,
           label='Diminishing returns threshold (ΔRMSE/ΔN < 0.005)')
ax.set_xticks(range(len(mid_sizes)))
ax.set_xticklabels([f'{int(a)}→{int(b)}' for a, b in zip(SIZES[:-1], SIZES[1:])],
                   rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Training Size Transition', fontsize=12, fontweight='bold')
ax.set_ylabel('ΔRMSE / ΔN  (negative = improvement per sample)',
              fontsize=12, fontweight='bold')
ax.set_title('Marginal Gain per Additional Sample\n'
             '(Diminishing Returns Analysis)',
             fontsize=13, fontweight='bold')
for bar_i, (b, v) in enumerate(zip(bars, marginal_rate)):
    ax.text(b.get_x() + b.get_width()/2,
            v - 0.0003 if v < 0 else v + 0.0003,
            f'{v:.4f}', ha='center', fontsize=8, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, ls='--', axis='y')

plt.suptitle(
    f'GP Matérn-0.5 — Convergence & Diminishing Returns (v4)  |  seed={RANDOM_STATE}',
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '9_powerlaw_marginal_v4.png'), **SAVE_KW)
plt.close()
print("    ✓ 图9 已保存")

# ── 图10: Bootstrap CI + Area Between Curves ─────────────────
print("  绘制 图10: Bootstrap CI + ABC...")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左：Bootstrap CI band
ax = axes[0]
ax.plot(SIZES, val_rmse_arr, 'o-', color=C_PINK, lw=3, ms=9,
        label='Val RMSE (observed mean)', zorder=3)
ax.fill_between(SIZES, boot_ci_lo, boot_ci_hi,
                alpha=0.30, color=C_PINK,
                label='Bootstrap 95% CI\n(1000 resamples)')
ax.plot(SIZES, lc_df['Train_RMSE_Mean'], 's-', color=C_BLUE, lw=3, ms=9,
        label='Train RMSE (observed mean)', zorder=3)
ax.fill_between(SIZES,
                lc_df['Train_RMSE_Mean'] - lc_df['Train_RMSE_Std'],
                lc_df['Train_RMSE_Mean'] + lc_df['Train_RMSE_Std'],
                alpha=0.20, color=C_BLUE)
ax.set_xlabel('Training Set Size', fontsize=12, fontweight='bold')
ax.set_ylabel('RMSE', fontsize=12, fontweight='bold')
ax.set_title('Learning Curve with Bootstrap 95% CI\n'
             '(Quantifies statistical uncertainty of the curve itself)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, framealpha=0.9)
ax.set_xticks(SIZES)
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, ls='--')

# 右：Area Between Curves
ax = axes[1]
ax.fill_between(SIZES,
                lc_df['Train_RMSE_Mean'], val_rmse_arr,
                where=(val_rmse_arr >= lc_df['Train_RMSE_Mean']),
                alpha=0.35, color=C_RED,
                label=f'Generalization gap area\n(ABC = {abc_total:.4f})')
ax.plot(SIZES, val_rmse_arr, 'o-', color=C_PINK, lw=3, ms=9,
        label='Val RMSE')
ax.plot(SIZES, lc_df['Train_RMSE_Mean'], 's-', color=C_BLUE, lw=3, ms=9,
        label='Train RMSE')

# Cumulative ABC
cumulative_abc = []
for i in range(1, N_SIZES):
    seg = np.trapz(np.maximum(rmse_gap_arr[:i+1], 0), SIZES[:i+1])
    cumulative_abc.append(seg)
ax2 = ax.twinx()
ax2.plot(SIZES[1:], cumulative_abc, ':', color=C_AMBER, lw=2.5,
         label='Cumulative ABC')
ax2.set_ylabel('Cumulative Area Between Curves', fontsize=11,
               fontweight='bold', color=C_AMBER)

ax.set_xlabel('Training Set Size', fontsize=12, fontweight='bold')
ax.set_ylabel('RMSE', fontsize=12, fontweight='bold')
ax.set_title(f'Area Between Curves (Generalization Cost)\n'
             f'Total ABC = {abc_total:.4f}  |  '
             f'(smaller = better generalization)',
             fontsize=13, fontweight='bold')
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labs1 + labs2, fontsize=10, framealpha=0.9)
ax.set_xticks(SIZES)
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, ls='--')

plt.suptitle(
    f'GP Matérn-0.5 — Bootstrap CI & Area Between Curves (v4)  |  seed={RANDOM_STATE}',
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '10_bootstrap_CI_ABC_v4.png'), **SAVE_KW)
plt.close()
print("    ✓ 图10 已保存")

# ── 图11: 核超参数演变 + σ/RMSE 比值 ─────────────────────────
print("  绘制 图11: 核超参数演变 + σ/RMSE 比值...")

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# (0,0): Length-scale
ax = axes[0][0]
if kp_df['LengthScale_Mean'].notna().any():
    ax.plot(kp_df['Train_Size'], kp_df['LengthScale_Mean'],
            'o-', color=C_BLUE, lw=3, ms=9, label='Length-scale (mean)')
    ax.fill_between(kp_df['Train_Size'],
                    kp_df['LengthScale_Mean'] - kp_df['LengthScale_Std'],
                    kp_df['LengthScale_Mean'] + kp_df['LengthScale_Std'],
                    alpha=0.2, color=C_BLUE)
    ax.set_title('Kernel Length-Scale vs Train Size\n'
                 '(↓ smaller → more local, ↑ larger → smoother)',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Length-scale', fontsize=11, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Length-scale not extractable\nfrom this kernel structure',
            ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.set_title('Kernel Length-Scale', fontsize=12, fontweight='bold')
ax.set_xlabel('Training Set Size', fontsize=11, fontweight='bold')
ax.set_xticks(SIZES); ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, ls='--')
ax.legend(fontsize=10)

# (0,1): Log Marginal Likelihood
ax = axes[0][1]
if kp_df['LML_Mean'].notna().any():
    ax.plot(kp_df['Train_Size'], kp_df['LML_Mean'],
            's-', color=C_GREEN, lw=3, ms=9,
            label='Log Marginal Likelihood (mean)')
    ax.fill_between(kp_df['Train_Size'],
                    kp_df['LML_Mean'] - kp_df['LML_Std'],
                    kp_df['LML_Mean'] + kp_df['LML_Std'],
                    alpha=0.2, color=C_GREEN)
    ax.set_title('Log Marginal Likelihood vs Train Size\n'
                 '(Bayesian evidence; higher → better model-data fit)',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Log Marginal Likelihood', fontsize=11, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'LML not available', ha='center', va='center',
            transform=ax.transAxes, fontsize=12)
ax.set_xlabel('Training Set Size', fontsize=11, fontweight='bold')
ax.set_xticks(SIZES); ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, ls='--')
ax.legend(fontsize=10)

# (1,0): Noise Level
ax = axes[1][0]
if kp_df['NoiseLevel_Mean'].notna().any():
    ax.plot(kp_df['Train_Size'], kp_df['NoiseLevel_Mean'],
            '^-', color=C_RED, lw=3, ms=9, label='Noise Level (mean)')
    ax.fill_between(kp_df['Train_Size'],
                    kp_df['NoiseLevel_Mean'] - kp_df['NoiseLevel_Std'],
                    kp_df['NoiseLevel_Mean'] + kp_df['NoiseLevel_Std'],
                    alpha=0.2, color=C_RED)
    ax.set_title('GP Noise Level vs Train Size\n'
                 '(Optimized noise variance in kernel)',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Noise Level', fontsize=11, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Noise level not separately\nextractable from this kernel',
            ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.set_title('GP Noise Level', fontsize=12, fontweight='bold')
ax.set_xlabel('Training Set Size', fontsize=11, fontweight='bold')
ax.set_xticks(SIZES); ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, ls='--')
ax.legend(fontsize=10)

# (1,1): σ/RMSE ratio
ax = axes[1][1]
if kp_df['Sigma_RMSE_Ratio_Mean'].notna().any():
    ratio_arr = kp_df['Sigma_RMSE_Ratio_Mean'].values
    colors_r = [C_RED if v > 1.5 else (C_ORANGE if v > 0.8 else C_GREEN)
                for v in ratio_arr]
    ax.bar(range(N_SIZES), ratio_arr, color=colors_r, alpha=0.70,
           edgecolor='black', lw=1)
    ax.axhline(y=1.0, color='black', lw=2.5, ls='--',
               label='σ = RMSE (perfect calibration)')
    ax.axhline(y=1.5, color='red', lw=2, ls=':', alpha=0.8,
               label='σ/RMSE=1.5 (over-conservative)')
    ax.axhline(y=0.8, color='orange', lw=2, ls=':', alpha=0.8,
               label='σ/RMSE=0.8 (over-confident)')
    ax.axhspan(0.8, 1.2, alpha=0.08, color='green',
               label='Well-calibrated zone (0.8–1.2)')
    for i, (v, c) in enumerate(zip(ratio_arr, colors_r)):
        ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9,
                fontweight='bold')
    ax.set_xticks(range(N_SIZES))
    ax.set_xticklabels([f'n={s}' for s in SIZES], rotation=45, ha='right')
    ax.set_ylabel('σ / RMSE  (Uncertainty-Error Ratio)',
                  fontsize=11, fontweight='bold')
    ax.set_title('σ/RMSE Calibration Ratio vs Train Size\n'
                 '(1.0 = perfect; >1 = over-conservative; <1 = over-confident)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    f'GP Matérn-0.5 — Kernel Diagnostics (v4)  |  seed={RANDOM_STATE}',
    fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '11_kernel_params_sigma_ratio_v4.png'), **SAVE_KW)
plt.close()
print("    ✓ 图11 已保存")

# ── 图12: 综合 6-Panel Dashboard + Sample Requirement ─────────
print("  绘制 图12: 综合 Dashboard...")

fig = plt.figure(figsize=(26, 18))
gs  = gridspec.GridSpec(3, 4, figure=fig,
                         hspace=0.45, wspace=0.38)

panel_specs = [
    # (row, col_slice, metric_name, y_values, color, ylabel)
    (0, slice(0,1), 'Val RMSE',   val_rmse_arr,              C_PINK,   'Val RMSE'),
    (0, slice(1,2), 'Val MAE',    lc_df['Val_MAE_Mean'].values,  C_RED,   'Val MAE'),
    (0, slice(2,3), 'Val R²',     lc_df['Val_R2_Mean'].values,   C_PURPLE,'Val R²'),
    (1, slice(0,1), 'RMSE Gap',   rmse_gap_arr,              C_ORANGE, 'RMSE Gap\n(Val-Train)'),
    (1, slice(1,2), 'GP σ',       fig3_data['Mean_Uncertainty'].values, '#9B59B6', 'GP σ (mean)'),
    (1, slice(2,3), 'Val RMSE CV',lc_df['Val_RMSE_CV'].values,  C_AMBER,  'Val RMSE CV(%)'),
]

for (row, col, name, yvals, color, ylabel) in panel_specs:
    ax = fig.add_subplot(gs[row, col])
    ax.plot(SIZES, yvals, 'o-', color=color, lw=2.5, ms=7, zorder=3)
    # 拐点标注
    ax.axvline(x=elbow_size, color='gray', ls=':', lw=2, alpha=0.7)
    ax.scatter([elbow_size], [yvals[elbow_idx]], color='gold',
               s=150, zorder=5, edgecolors='black', lw=1.5,
               label=f'Elbow n={elbow_size}')
    ax.set_xlabel('Training Size', fontsize=10, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=10, fontweight='bold')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xticks(SIZES[::2])
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    ax.grid(True, alpha=0.3, ls='--')
    ax.legend(fontsize=8)
    if name == 'Val R²':
        ax.axhline(y=0, color='red', ls='--', lw=1.5, alpha=0.7)
    if name == 'Val RMSE CV':
        ax.axhline(y=10, color='green',  ls='--', lw=1.5, alpha=0.7)
        ax.axhline(y=30, color='orange', ls='--', lw=1.5, alpha=0.7)

# 第三行左3格：幂律曲线（大图）
ax_big = fig.add_subplot(gs[2, :2])
ax_big.plot(SIZES, val_rmse_arr, 'o-', color=C_PINK, lw=3, ms=9,
            label='Val RMSE (observed)', zorder=3)
ax_big.fill_between(SIZES, boot_ci_lo, boot_ci_hi,
                    alpha=0.25, color=C_PINK,
                    label='Bootstrap 95% CI')
if pl_fit_ok:
    n_dense2 = np.linspace(SIZES[0], max(SIZES[-1], 250), 500)
    ax_big.plot(n_dense2, power_law(n_dense2, a_val, b_val), '--',
                color=C_PURPLE, lw=2.5,
                label=f'Power-law: {a_val:.4f}·n^{b_val:.4f}')
ax_big.axvline(x=elbow_size, color='orange', ls=':', lw=2.5)
ax_big.scatter([elbow_size], [elbow_rmse], color='orange', s=200, zorder=5,
               edgecolors='black', lw=2, label=f'Elbow: n={elbow_size}')
ax_big.set_xlabel('Training Set Size', fontsize=12, fontweight='bold')
ax_big.set_ylabel('Val RMSE', fontsize=12, fontweight='bold')
ax_big.set_title('Power-Law Fit + Bootstrap CI (Summary)',
                 fontsize=13, fontweight='bold')
ax_big.legend(fontsize=10, framealpha=0.9)
ax_big.set_xticks(SIZES)
ax_big.tick_params(axis='x', rotation=45)
ax_big.grid(True, alpha=0.3, ls='--')

# 第三行右2格：Sample Requirement
ax_req = fig.add_subplot(gs[2, 2:])
if target_rmses is not None:
    ax_req.plot(required_ns, target_rmses, '-', color=C_TEAL, lw=3,
                label=f'Predicted n needed\n(Power-law: a={a_val:.4f}, b={b_val:.4f})')
    ax_req.axvline(x=SIZES[-1], color='blue', ls=':', lw=2.5,
                   label=f'Current dataset n={SIZES[-1]}')
    ax_req.axhline(y=val_rmse_arr[-1], color=C_PINK, ls='--', lw=2,
                   label=f'Current RMSE={val_rmse_arr[-1]:.4f}')
    # 目标50%改善
    target_50 = val_rmse_arr[-1] * 0.5
    if pl_fit_ok and b_val < 0 and target_50 > 0:
        try:
            n_for_50 = (target_50 / a_val) ** (1.0 / b_val)
            if 0 < n_for_50 < 10000:
                ax_req.axvline(x=n_for_50, color='red', ls=':', lw=2,
                               label=f'50% RMSE reduction → n≈{n_for_50:.0f}')
                ax_req.axhline(y=target_50, color='red', ls=':', lw=2)
        except Exception:
            pass
    ax_req.set_xlabel('Required Training Samples (n)', fontsize=12,
                      fontweight='bold')
    ax_req.set_ylabel('Target Val RMSE', fontsize=12, fontweight='bold')
    ax_req.set_title('Sample Requirement Prediction\n'
                     '(Target RMSE → n needed)',
                     fontsize=13, fontweight='bold')
    ax_req.legend(fontsize=10, framealpha=0.9)
    ax_req.grid(True, alpha=0.3, ls='--')
else:
    ax_req.text(0.5, 0.5, 'Power-law fit failed\n(insufficient data range)',
                ha='center', va='center', transform=ax_req.transAxes,
                fontsize=13)
    ax_req.set_title('Sample Requirement Prediction', fontsize=13,
                     fontweight='bold')

plt.suptitle(
    f'GP Matérn-0.5 — Comprehensive Learning Curve Dashboard (v4)\n'
    f'seed={RANDOM_STATE}  |  n_restarts={N_RESTARTS}  |  '
    f'Power-law: RMSE = {a_val:.4f}·n^{b_val:.4f}  |  '
    f'Elbow: n={elbow_size}  |  ABC={abc_total:.4f}',
    fontsize=15, fontweight='bold', y=1.01)
plt.savefig(os.path.join(OUTPUT_DIR, '12_comprehensive_dashboard_v4.png'),
            **SAVE_KW)
plt.close()
print("    ✓ 图12 已保存")

# ============================================================
# 步骤 8: 文字报告
# ============================================================
print("\n[8/8] 生成文字报告...")

report_path = os.path.join(OUTPUT_DIR, 'GP_Learning_Curve_v4_Report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("TOP1 GP 模型完整学习曲线分析报告 v4\n")
    f.write("Model: Gp_GP_Matern_0.5_K6_Comb17\n")
    f.write("=" * 80 + "\n\n")
    f.write("1. 分析配置\n" + "-"*60 + "\n")
    f.write(f"  特征: {', '.join(feature_names)}\n")
    f.write(f"  总样本: {len(y_all)}  验证集: {N_VAL} 样本\n")
    f.write(f"  训练集大小序列: {list(SIZES)}\n")
    f.write(f"  每个大小重复次数: {N_ITERATIONS}\n")
    f.write(f"  验证集种子: {RANDOM_STATE}\n\n")
    f.write("2. 核心学习曲线结果\n" + "-"*60 + "\n")
    f.write(f"  {'Size':>6} | {'Val RMSE':>10} | {'Val MAE':>9} | "
            f"{'Val R²':>8} | {'GP σ':>8} | {'CV%':>6} | {'σ/RMSE':>8}\n")
    f.write("  " + "-"*75 + "\n")
    for i, (_, row) in enumerate(lc_df.iterrows()):
        ratio = kp_df.iloc[i]['Sigma_RMSE_Ratio_Mean'] if i < len(kp_df) else np.nan
        f.write(f"  {int(row['Train_Size']):>6} | "
                f"{row['Val_RMSE_Mean']:>10.4f} | "
                f"{row['Val_MAE_Mean']:>9.4f} | "
                f"{row['Val_R2_Mean']:>8.4f} | "
                f"{row['GP_Std_Mean']:>8.4f} | "
                f"{row['Val_RMSE_CV']:>6.1f} | "
                f"{ratio:>8.3f}\n")
    f.write("\n3. 改善统计\n" + "-"*60 + "\n")
    f.write(f"  Val RMSE: {lc_df['Val_RMSE_Mean'].iloc[0]:.4f} → "
            f"{lc_df['Val_RMSE_Mean'].iloc[-1]:.4f}  ({rmse_improv:.1f}%↓)\n")
    f.write(f"  Val MAE:  {lc_df['Val_MAE_Mean'].iloc[0]:.4f} → "
            f"{lc_df['Val_MAE_Mean'].iloc[-1]:.4f}  ({mae_improv:.1f}%↓)\n")
    f.write(f"  Val R²:   {r2_init:.4f} → {r2_final:.4f}  "
            f"({'+'if r2_change>=0 else ''}{r2_change:.4f})\n\n")
    f.write("4. 收敛分析\n" + "-"*60 + "\n")
    f.write(f"  幂律拟合: RMSE = {a_val:.6f} × n^{b_val:.6f}\n")
    f.write(f"  收敛速率指数 b = {b_val:.6f}  (|b| 越大收敛越快)\n")
    f.write(f"  最优拐点 (Elbow): n={elbow_size}, Val RMSE={elbow_rmse:.4f}\n")
    f.write(f"  泛化代价积分 (ABC): {abc_total:.4f}\n\n")
    f.write("5. GP 专属诊断\n" + "-"*60 + "\n")
    if not picp_df.empty:
        f.write(f"  n={SIZES[-1]} 下 95% PI 实际覆盖率: "
                f"{picp_df['PICP_Mean'].iloc[-1]*100:.1f}%"
                f" (标准: 95%)\n")
        f.write(f"  n={SIZES[-1]} 下 MPIW: {picp_df['MPIW_Mean'].iloc[-1]:.4f}\n")
    if kp_df['Sigma_RMSE_Ratio_Mean'].notna().any():
        final_ratio = kp_df['Sigma_RMSE_Ratio_Mean'].iloc[-1]
        calib_desc  = ('过度保守' if final_ratio > 1.2
                       else ('过度自信' if final_ratio < 0.8
                             else '校准良好'))
        f.write(f"  n={SIZES[-1]} 下 σ/RMSE = {final_ratio:.3f}  ({calib_desc})\n\n")
    f.write("6. 输出文件汇总\n" + "-"*60 + "\n")
    for i in range(1, 13):
        f.write(f"  图{i:02d}: ")
    f.write("\n  Excel: GP_Learning_Curve_v4_Complete_Data.xlsx (17个工作表)\n")
    f.write(f"\n报告生成时间: {pd.Timestamp.now()}\n")
    f.write("=" * 80 + "\n")

# ============================================================
# 汇总输出
# ============================================================
print("\n" + "=" * 80)
print("✅ GP 模型学习曲线分析 v4 完成！")
print("=" * 80)
print(f"\n📁 输出目录: {OUTPUT_DIR}")
print(f"\n📊 可视化输出（12张图）:")
files_out = [
    "1_learning_curve_RMSE_MAE_R2_v4.png      ← 原 v3 Fig1（更新）",
    "2_generalization_gaps_v4.png             ← 原 v3 Fig2（更新）",
    "3_GP_confidence_interval_v4.png          ← 原 v3 Fig3（更新）",
    "4_box_violin_variance_v4.png             ← [NEW] Box+Violin 方差诊断",
    "5_CV_stability_v4.png                    ← [NEW] CV(%) 稳定性曲线",
    "6_residual_distribution_v4.png           ← [NEW] 残差分布演变",
    "7_calibration_PI_coverage_v4.png         ← [NEW] 校准曲线 + PI覆盖率",
    "8_PICP_MPIW_v4.png                       ← [NEW] PICP-MPIW 联合图",
    "9_powerlaw_marginal_v4.png               ← [NEW] 幂律拟合 + 边际收益",
    "10_bootstrap_CI_ABC_v4.png               ← [NEW] Bootstrap CI + ABC",
    "11_kernel_params_sigma_ratio_v4.png      ← [NEW] 核超参数 + σ/RMSE",
    "12_comprehensive_dashboard_v4.png        ← [NEW] 综合 Dashboard",
]
for f_name in files_out:
    print(f"  • {f_name}")

print(f"\n📋 Excel: GP_Learning_Curve_v4_Complete_Data.xlsx（17个工作表）")
print(f"\n🎯 关键结论:")
print(f"  Val RMSE:  {lc_df['Val_RMSE_Mean'].iloc[-1]:.4f}  ({rmse_improv:.1f}%↓ vs n={SIZES[0]})")
print(f"  Val R²:    {r2_final:.4f}")
print(f"  幂律指数:   b = {b_val:.4f}")
print(f"  最优拐点:   n = {elbow_size}  (Val RMSE={elbow_rmse:.4f})")
print(f"  泛化代价:   ABC = {abc_total:.4f}")
print("=" * 80)

TOP1 GP 模型 (Gp_GP_Matern_0.5_K6_Comb17) 完整学习曲线分析程序 v4
v4 新增：方差/稳定性 | 预测质量 | 收敛分析 | GP专属 | 实践指导（共12张图）

[1/8] 加载 TOP1 GP 模型和数据...
  ✓ 模型: GP_Matern_0.5
  ✓ 特征 (6个): ['deltaP_热导率W/(mk)', 'x', 'Ec', 'Ω', 'deltaP_E13 Electron affinity', 'ΔSmix']
  ✓ 核函数: 1.07**2 * Matern(length_scale=2.29, nu=0.5) + WhiteKernel(noise_level=0.252)
  ✓ 有效样本: 202,  KQ ∈ [3.400, 19.800]

[2/8] 固定验证集划分（seed=2023）...
  ✓ 训练集: 161  验证集: 41
  ✓ 验证集 KQ 均值: 10.838,  序列: [5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 130, 161]

[3/8] 学习曲线主循环（RMSE/MAE/R² + GP专属信息）...
  n=   5 ... ValRMSE=3.2322±0.3117  R²=-0.044  σ=2.8165
  n=  10 ... ValRMSE=3.0903±0.3329  R²=0.043  σ=2.6929
  n=  15 ... ValRMSE=2.7203±0.3423  R²=0.256  σ=2.5450
  n=  20 ... ValRMSE=2.6909±0.2933  R²=0.274  σ=2.6049
  n=  25 ... ValRMSE=2.6279±0.2916  R²=0.308  σ=2.5044
  n=  30 ... ValRMSE=2.4162±0.2190  R²=0.417  σ=2.5311
  n=  40 ... ValRMSE=2.3806±0.2274  R²=0.434  σ=2.5010
  n=  50 ... ValRMSE=2.1898±0.2060  R²=0.521  σ=2.3168
  n=  60 ... ValRMSE